# Odds history: starter questions

These cells answer two example questions from the data the bot saves every day. They only **read** `data/odds.db`.

Before running: build the database with the odds **and** the final scores (see `analysis/README.md`).

> **Read this first.** The lines here are the bot's *morning* lines (pulled around 7:45 to 8 AM), not closing lines, and a game only appears once it has both pre-game odds and a final score. Early in a season the sample is small, so treat small differences as noise.

In [ ]:
import pandas as pd
import load

games = load.game_lines()
print(len(games), "finished games with pre-game lines")
games.groupby("sport_key").size().sort_values(ascending=False)

## How often do home underdogs win?

`home_underdog` is 1 when the sportsbooks (on average) gave the home team a lower chance to win than the visitors.

In [ ]:
dogs = games[games["home_underdog"] == 1]

by_sport = (dogs.groupby("sport_key")
            .agg(games=("event_id", "count"),
                 home_wins=("home_won", "sum"),
                 market_said=("home_prob", "mean"),     # the average win chance the odds implied
                 actual=("home_won", "mean")))          # how often they really won
by_sport.sort_values("games", ascending=False)

In [ ]:
nhl = dogs[dogs["sport_key"] == "icehockey_nhl"]
if len(nhl):
    print(f"NHL home underdogs won {nhl['home_won'].mean():.1%} of {len(nhl)} games "
          f"(the odds implied about {nhl['home_prob'].mean():.1%})")
else:
    print("No NHL home underdogs with results yet.")

## Do NBA totals go over more on back-to-backs?

`load.with_rest_days` adds `home_b2b` and `away_b2b` (True when that team also played the previous calendar day, Central time). Rest days come from the games in the database, so the first games after collection began can look rested when they were not.

In [ ]:
with_rest = load.with_rest_days(games)
nba = with_rest[(with_rest["sport_key"] == "basketball_nba") & with_rest["went_over"].notna()].copy()
nba["any_b2b"] = nba["home_b2b"].fillna(False).astype(bool) | nba["away_b2b"].fillna(False).astype(bool)

(nba.groupby("any_b2b")["went_over"]
    .agg(games="count", over_rate="mean"))

## Notes before trusting an answer

- **Pushes** (the score lands exactly on the total) have `went_over` empty and are left out.
- **Soccer cup games** can include extra time; check the `status` column (`FINAL_AET`, `FINAL_PEN`) and filter if you want 90 minutes only.
- **Sample size.** A few dozen games cannot separate a real effect from luck. Wait for a few hundred.
- The `pulls` and `events` tables show which games and days are missing from the data.